In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI HASIL UNDUHAN WAVEFORM
- Jumlah file, ukuran, distribusi stasiun
- Komponen yang tersedia (Z, N, E)
- Kualitas sinyal (SNR) untuk sampel
- Deteksi file corrupt
"""

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_juli_hybrid_katalog_2026"
OUTPUT_DIR = "investigasi_waveform"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# Parameter untuk sampel SNR
SAMPLE_SIZE = 500  # ambil 500 file acak untuk analisis SNR
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0

# =============================================
# 2. BACA FILE .mseed
# =============================================

print("="*70)
print("📊 INVESTIGASI HASIL UNDUHAN WAVEFORM")
print("="*70)

print(f"\n📂 Membaca direktori: {WAVEFORM_DIR}")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
# Filter file ._ (macOS metadata)
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed valid: {len(files):,}")

if len(files) == 0:
    print("❌ Tidak ada file .mseed ditemukan!")
    exit()

# =============================================
# 3. EKSTRAK METADATA DARI NAMA FILE
# =============================================

print("\n📊 Ekstrak metadata dari nama file...")

def parse_filename(filename):
    """Ekstrak network, station, dan event_id dari nama file."""
    stem = filename.stem
    parts = stem.split('_')
    
    if len(parts) >= 3:
        network = parts[0]
        station = parts[1]
        event_id = '_'.join(parts[2:])
    else:
        network = 'UNK'
        station = 'UNK'
        event_id = stem
    
    return {
        'file': filename.name,
        'network': network,
        'station': station,
        'event_id': event_id,
        'size_kb': filename.stat().st_size / 1024
    }

metadata = []
for f in files:
    meta = parse_filename(f)
    metadata.append(meta)

df_meta = pd.DataFrame(metadata)
print(f"✅ Metadata diekstrak dari {len(df_meta):,} file")

# =============================================
# 4. STATISTIK DASAR
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK DASAR")
print("="*70)

print(f"Total file: {len(df_meta):,}")
print(f"Total storage: {df_meta['size_kb'].sum() / (1024*1024):.2f} GB")
print(f"Rata-rata ukuran: {df_meta['size_kb'].mean():.1f} KB")
print(f"Median ukuran: {df_meta['size_kb'].median():.1f} KB")
print(f"Min ukuran: {df_meta['size_kb'].min():.1f} KB")
print(f"Max ukuran: {df_meta['size_kb'].max():.1f} KB")

# =============================================
# 5. DISTRIBUSI STASIUN
# =============================================

print("\n📡 20 STASIUN TERBANYAK:")
station_counts = df_meta['station'].value_counts()
for sta, count in station_counts.head(20).items():
    pct = count / len(df_meta) * 100
    print(f"  {sta}: {count:,} ({pct:.1f}%)")

print(f"\n📡 Total stasiun unik: {df_meta['station'].nunique():,}")

# =============================================
# 6. DISTRIBUSI JARINGAN
# =============================================

print("\n📡 10 JARINGAN TERBANYAK:")
network_counts = df_meta['network'].value_counts()
for net, count in network_counts.head(10).items():
    pct = count / len(df_meta) * 100
    print(f"  {net}: {count:,} ({pct:.1f}%)")

print(f"\n📡 Total jaringan unik: {df_meta['network'].nunique():,}")

# =============================================
# 7. CEK KOMPONEN (Z, N, E) UNTUK SAMPEL
# =============================================

print("\n📊 CEK KOMPONEN (Z, N, E) - 500 file sampel...")

def check_components(file_path):
    """Cek komponen yang tersedia dalam file .mseed."""
    try:
        st = read(str(file_path))
        channels = [tr.stats.channel for tr in st]
        has_z = any(ch.endswith('Z') for ch in channels)
        has_n = any(ch.endswith('N') for ch in channels)
        has_e = any(ch.endswith('E') for ch in channels)
        return {
            'has_z': has_z,
            'has_n': has_n,
            'has_e': has_e,
            'channels': channels,
            'n_traces': len(st)
        }
    except Exception as e:
        return {
            'has_z': False,
            'has_n': False,
            'has_e': False,
            'channels': [],
            'n_traces': 0,
            'error': str(e)
        }

# Ambil 500 file acak
sample_files = np.random.choice(files, min(SAMPLE_SIZE, len(files)), replace=False)
component_data = []

for f in tqdm(sample_files, desc="Cek komponen"):
    comp = check_components(f)
    comp['file'] = f.name
    component_data.append(comp)

df_comp = pd.DataFrame(component_data)

print(f"\n✅ File dengan komponen Z: {df_comp['has_z'].sum()}/{len(df_comp)} ({df_comp['has_z'].sum()/len(df_comp)*100:.1f}%)")
print(f"✅ File dengan komponen N: {df_comp['has_n'].sum()}/{len(df_comp)} ({df_comp['has_n'].sum()/len(df_comp)*100:.1f}%)")
print(f"✅ File dengan komponen E: {df_comp['has_e'].sum()}/{len(df_comp)} ({df_comp['has_e'].sum()/len(df_comp)*100:.1f}%)")

# File tanpa Z
no_z = df_comp[~df_comp['has_z']]
if len(no_z) > 0:
    print(f"\n⚠️ {len(no_z)} file TANPA komponen Z (contoh):")
    for _, row in no_z.head(5).iterrows():
        print(f"  - {row['file']} (channels: {row['channels']})")

# File dengan 3 komponen
three_comp = df_comp[(df_comp['has_z']) & (df_comp['has_n']) & (df_comp['has_e'])]
print(f"\n✅ File dengan 3 komponen (Z, N, E): {len(three_comp)}/{len(df_comp)} ({len(three_comp)/len(df_comp)*100:.1f}%)")

# =============================================
# 8. CEK FILE CORRUPT / TERLALU KECIL
# =============================================

print("\n📊 CEK FILE CORRUPT / TERLALU KECIL...")

# File dengan ukuran < 1 KB dianggap corrupt
corrupt_files = df_meta[df_meta['size_kb'] < 1]
print(f"✅ File dengan ukuran < 1 KB (corrupt): {len(corrupt_files):,} ({len(corrupt_files)/len(df_meta)*100:.2f}%)")

# File dengan ukuran < 10 KB (mungkin tidak lengkap)
small_files = df_meta[(df_meta['size_kb'] >= 1) & (df_meta['size_kb'] < 10)]
print(f"✅ File dengan ukuran 1-10 KB (mungkin tidak lengkap): {len(small_files):,} ({len(small_files)/len(df_meta)*100:.2f}%)")

# =============================================
# 9. SIMPAN HASIL
# =============================================

df_meta.to_csv(f"{OUTPUT_DIR}/file_metadata.csv", index=False)
df_comp.to_csv(f"{OUTPUT_DIR}/component_analysis.csv", index=False)
print(f"\n✅ Metadata tersimpan: {OUTPUT_DIR}/file_metadata.csv")
print(f"✅ Komponen tersimpan: {OUTPUT_DIR}/component_analysis.csv")

# =============================================
# 10. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN INVESTIGASI")
print("="*70)
print(f"Total file: {len(df_meta):,}")
print(f"Total storage: {df_meta['size_kb'].sum() / (1024*1024):.2f} GB")
print(f"File dengan 3 komponen: {len(three_comp)}/{len(df_comp)} ({len(three_comp)/len(df_comp)*100:.1f}%)")
print(f"File corrupt (<1 KB): {len(corrupt_files):,}")
print(f"Stasiun terbanyak: {station_counts.index[0]} ({station_counts.iloc[0]:,} file)")
print(f"Jaringan terbanyak: {network_counts.index[0]} ({network_counts.iloc[0]:,} file)")
print("="*70)

📊 INVESTIGASI HASIL UNDUHAN WAVEFORM

📂 Membaca direktori: /Volumes/Extreme SSD/unduhan_juli_hybrid_katalog_2026
✅ Total file .mseed valid: 18,804

📊 Ekstrak metadata dari nama file...
✅ Metadata diekstrak dari 18,804 file

📊 STATISTIK DASAR
Total file: 18,804
Total storage: 0.42 GB
Rata-rata ukuran: 23.5 KB
Median ukuran: 19.5 KB
Min ukuran: 0.5 KB
Max ukuran: 207.0 KB

📡 20 STASIUN TERBANYAK:
  TNTI: 4,076 (21.7%)
  PMG: 2,599 (13.8%)
  GENI: 1,367 (7.3%)
  MNAI: 1,182 (6.3%)
  GSI: 1,039 (5.5%)
  BNDI: 877 (4.7%)
  LHMI: 858 (4.6%)
  SAUI: 841 (4.5%)
  FAKI: 778 (4.1%)
  LUWI: 672 (3.6%)
  PLAI: 580 (3.1%)
  TOLI2: 512 (2.7%)
  JAGI: 475 (2.5%)
  BBJI: 452 (2.4%)
  SANI: 443 (2.4%)
  SOEI: 301 (1.6%)
  MMRI: 284 (1.5%)
  CISI: 281 (1.5%)
  BKNI: 266 (1.4%)
  BB02: 239 (1.3%)

📡 Total stasiun unik: 32

📡 10 JARINGAN TERBANYAK:
  GE: 18,462 (98.2%)
  7G: 341 (1.8%)
  XN: 1 (0.0%)

📡 Total jaringan unik: 3

📊 CEK KOMPONEN (Z, N, E) - 500 file sampel...


Cek komponen: 100%|██████████| 500/500 [00:01<00:00, 423.63it/s]


✅ File dengan komponen Z: 499/500 (99.8%)
✅ File dengan komponen N: 426/500 (85.2%)
✅ File dengan komponen E: 427/500 (85.4%)

⚠️ 1 file TANPA komponen Z (contoh):
  - GE_TNTI_20150705_013608.mseed (channels: ['BHE'])

✅ File dengan 3 komponen (Z, N, E): 426/500 (85.2%)

📊 CEK FILE CORRUPT / TERLALU KECIL...
✅ File dengan ukuran < 1 KB (corrupt): 1 (0.01%)
✅ File dengan ukuran 1-10 KB (mungkin tidak lengkap): 384 (2.04%)

✅ Metadata tersimpan: investigasi_waveform/file_metadata.csv
✅ Komponen tersimpan: investigasi_waveform/component_analysis.csv

📊 RINGKASAN INVESTIGASI
Total file: 18,804
Total storage: 0.42 GB
File dengan 3 komponen: 426/500 (85.2%)
File corrupt (<1 KB): 1
Stasiun terbanyak: TNTI (4,076 file)
Jaringan terbanyak: GE (18,462 file)
